# CDC from scratch in PySpark

Lakeflow's `create_auto_cdc_flow` is a black box on Databricks. This notebook rebuilds its **rules** with plain PySpark, so you can see *why* each argument exists. It runs locally: no Databricks, no Delta.

**What it is / isn't.** It reproduces the behaviour (order by `seq`, keep tombstones, ignore stale events). It is not Databricks' internal implementation, and SCD2 is left to the pipeline file.

**Folder layout** (next to this notebook):
```
cdc_from_scratch.ipynb
customers_cdc/            4 JSON files (seed + 3 weekly batches)
cdc_ground_truth.json     the answer key
```
**Requirements:** Python 3.9+, Java 17, `pip install pyspark pandas`.

**The plan:** detect the five planted problems (C1 to C5) with numbers, then replay the feed batch by batch and watch each wrong strategy miss the truth of **8,170** live customers.

In [1]:
import json, glob
from pyspark.sql import SparkSession, Window, functions as F

spark = (SparkSession.builder.master("local[*]").appName("cdc-from-scratch")
         .config("spark.sql.shuffle.partitions", 4).config("spark.ui.enabled", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

FEED = "customers_cdc"
files = sorted(glob.glob(f"{FEED}/*.json"))      # arrival order = file order
COLS = ["customer_id", "first_name", "last_name", "email", "signup_date", "country", "marketing_opt_in"]
truth = json.load(open("cdc_ground_truth.json"))
print([f.split("_")[-1] for f in files])

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/04 11:39:44 WARN Utils: Your hostname, vm, resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/11/04 11:39:44 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/04 11:39:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


['2025-09-29.json', '2025-10-06.json', '2025-10-13.json', '2025-10-20.json']


## 1. Load the events
Each file is one arrival batch. `arrival` is the file index (0 = seed). `seq` is the event order the source system assigned: the only trustworthy clock.

In [2]:
def load(i, f):
    return (spark.read.json(f)
            .select(*COLS, "op", "seq", "event_ts")
            .withColumn("is_deleted", F.col("op") == "D")
            .withColumn("arrival", F.lit(i)))

batches = [load(i, f) for i, f in enumerate(files)]
events = batches[0]
for b in batches[1:]:
    events = events.unionByName(b)
events = events.cache()
print("lines:", events.count())
events.filter("arrival = 1").orderBy("seq").show(5, truncate=False)

lines: 9729


+-----------+----------+---------+------------------------------+-----------+-------+----------------+---+---+-------------------+----------+-------+
|customer_id|first_name|last_name|email                         |signup_date|country|marketing_opt_in|op |seq|event_ts           |is_deleted|arrival|
+-----------+----------+---------+------------------------------+-----------+-------+----------------+---+---+-------------------+----------+-------+
|CUST007195 |Benjamin  |Miller   |nancylopez@example.com        |2025-05-07 |US     |false           |U  |3  |2025-09-29T00:07:08|false     |1      |
|CUST004237 |Andrea    |Chan     |u4854_aaronramirez@example.org|2025-08-24 |US     |false           |U  |4  |2025-09-29T00:30:07|false     |1      |
|CUST005057 |Johnathan |Nelson   |u9206_u5908_chess@example.org |2025-07-20 |UK     |false           |U  |5  |2025-09-29T00:40:02|false     |1      |
|CUST008146 |New145    |Customer |new145@example.com            |2025-09-29 |NG     |true           

## 2. Detect the five problems
Each count should match the answer key.

In [3]:
real = events.filter("arrival > 0")
uniq = real.dropDuplicates(["seq"])

# C4 duplicates: same seq emitted more than once
c4 = real.groupBy("seq").count().filter("count > 1").count()

# C1 out of order: the week an event belongs to (from event_ts) vs the file it arrived in
implied = F.greatest(F.lit(1), F.least(F.lit(3),
    F.floor(F.datediff(F.to_date("event_ts"), F.lit("2025-09-29").cast("date")) / 7).cast("int") + 1))
c1 = uniq.withColumn("implied", implied).filter(F.col("arrival") > F.col("implied")).count()

# C5 deletes carry only the key
c5 = real.filter("op = 'D'").filter(F.col("email").isNull()).dropDuplicates(["seq"]).count()

# C2 stale update: an update with a LOWER seq than a delete of the same customer, arriving after that delete
d = uniq.filter("op = 'D'").select("customer_id", F.col("seq").alias("d_seq"), F.col("arrival").alias("d_arr"))
u = uniq.filter("op = 'U'").select("customer_id", "seq", "arrival")
j = u.join(d, "customer_id").filter("seq < d_seq")
c2_later = j.filter("arrival > d_arr").count()
c2_all = j.count()

# C3 re-creation: an insert with a higher seq than a delete of the same customer
i_ = uniq.filter("op = 'I'").select("customer_id", F.col("seq").alias("i_seq"))
c3 = d.join(i_, "customer_id").filter("i_seq > d_seq").select("customer_id").distinct().count()

print(f"C1 out-of-order events : {c1}   (key: 141)")
print(f"C2 stale after delete  : {c2_all} total, {c2_later} in a later batch   (key: 30 / 28)")
print(f"C3 re-created          : {c3}   (key: {truth['recreated_after_delete']})")
print(f"C4 duplicate seqs      : {c4}   (key: {truth['duplicate_lines_emitted']})")
print(f"C5 key-only deletes    : {c5}   (key: {truth['events_by_op']['D']})")

C1 out-of-order events : 141   (key: 141)
C2 stale after delete  : 30 total, 28 in a later batch   (key: 30 / 28)
C3 re-created          : 20   (key: 20)
C4 duplicate seqs      : 29   (key: 29)
C5 key-only deletes    : 150   (key: 150)


## 3. The truth, in one shot
If you could see every event at once, the answer is trivial: highest `seq` per customer wins, drop customers whose winner is a delete. A streaming pipeline can't do this, because it only sees one batch at a time. That's the whole difficulty.

In [4]:
def latest(df, order):
    w = Window.partitionBy("customer_id").orderBy(*order)
    return df.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")

one_shot = latest(events, [F.col("seq").desc()])
TRUE_LIVE = one_shot.filter("not is_deleted")
print("live customers:", TRUE_LIVE.count(), "| expected:", truth["expected_scd1_final_rows"])

def score(live_df):
    live_df = live_df.filter("not is_deleted")
    cols = COLS
    wrong_or_missing = TRUE_LIVE.select(*cols).exceptAll(live_df.select(*cols)).count()
    extra = live_df.select(*cols).exceptAll(TRUE_LIVE.select(*cols)).count()
    return live_df.count(), wrong_or_missing, extra

live customers: 8170 | expected: 8170


## 4. Replay batch by batch

`apply_batch` is the heart of it: union the state with the new batch, rank each customer's events, keep the winner. Three knobs:

- `order`: what decides the winner (`seq`, or arrival order)
- `tomb_batches`: how many batches a delete is remembered before it is garbage-collected
- `delete_as_row`: treat a delete as ordinary data (the C5 mistake)

In [5]:
def apply_batch(state, batch, i, order, tomb_batches=None, delete_as_row=False):
    if delete_as_row:
        batch = batch.withColumn("is_deleted", F.lit(False))
    combined = batch if state is None else state.unionByName(batch)
    new = latest(combined, order)
    if tomb_batches is not None:   # None = keep tombstones forever
        age = F.lit(i) - F.col("arrival")
        new = new.filter(~(F.col("is_deleted") & (age >= tomb_batches)))
    return new.cache()

def replay(order, tomb_batches=None, delete_as_row=False, verbose=False):
    state = None
    for i, b in enumerate(batches):
        state = apply_batch(state, b, i, order, tomb_batches, delete_as_row)
        if verbose:
            print(f"  after batch {i}: live={state.filter('not is_deleted').count():>5}  tombstones={state.filter('is_deleted').count()}")
    return state

SEQ = [F.col("seq").desc()]
ARRIVAL = [F.col("arrival").desc(), F.col("seq").desc()]

### Case A: naive. Last file wins, deletes are forgotten
What a hand-rolled "upsert whatever arrived last" job does.

In [6]:
A = replay(ARRIVAL, tomb_batches=0)
print("live, wrong, extra:", score(A), "| truth:", truth["expected_scd1_final_rows"])

live, wrong, extra: (8197, 7, 34) | truth: 8170


### Case B: order by `seq`, but forget deletes immediately
Fixes C1 and C4. A delete still vanishes the moment it is applied, so a stale update (C2) arriving later looks like a brand new customer.

In [7]:
B = replay(SEQ, tomb_batches=0, verbose=True)
live, wrong, extra = score(B)
print("live:", live, "| resurrected (extra) rows:", extra, "| stale updates in a later batch:", c2_later)

  after batch 0: live= 8000  tombstones=0


  after batch 1: live= 8028  tombstones=0


  after batch 2: live= 8058  tombstones=0


  after batch 3: live= 8198  tombstones=0


live: 8198 | resurrected (extra) rows: 28 | stale updates in a later batch: 28


### Case C: order by `seq` and keep tombstones
This is what `apply_as_deletes` plus the sequence check do. The delete stays in state as a row with `is_deleted = true` and its `seq`, so a lower-`seq` update loses to it.

In [8]:
Cc = replay(SEQ, tomb_batches=None, verbose=True)
print("live, wrong, extra:", score(Cc))
print("tombstones held:", Cc.filter("is_deleted").count(), "| key: deleted and not re-created =", truth["expected_deleted_and_not_recreated"])

  after batch 0: live= 8000  tombstones=0


  after batch 1: live= 8028  tombstones=61


  after batch 2: live= 8058  tombstones=128


  after batch 3: live= 8170  tombstones=130


live, wrong, extra: (8170, 0, 0)
tombstones held: 130 | key: deleted and not re-created = 130


### Case D: the tombstone window
Keeping tombstones forever grows state. Lakeflow keeps them for a retention window (default 2 days). Here the window is counted in batches. How long is long enough?

In [9]:
print("window  live   resurrected")
for n in [0, 1, 2, 3, 4]:
    s = replay(SEQ, tomb_batches=n)
    live, wrong, extra = score(s)
    print(f"{n:>5}  {live:>5}  {extra:>5}")

window  live   resurrected


    0   8198     28


    1   8183     13


    2   8170      0


    3   8170      0


    4   8170      0


Read the table: a window of 0 resurrects every stale update. Once the window is longer than the **longest delay between a delete and its latest straggler**, the answer is exact. On Databricks that threshold is `pipelines.cdc.tombstoneGCThresholdInSeconds`. The mechanics here are my model of it; check the exact clock semantics in the docs before relying on them.

### Case E: the C5 mistake: a delete treated as an update
A delete row has null non-key columns. If you write it like any other row, you keep a customer with nulls instead of removing them.

In [10]:
E = replay(SEQ, tomb_batches=None, delete_as_row=True)
nulled = E.filter("email is null").count()
print("rows with null email (deleted customers kept as ghosts):", nulled, "| live count reported:", E.count())

rows with null email (deleted customers kept as ghosts): 130 | live count reported: 8300


## 5. Bridge to the real thing

| What you wrote here | Lakeflow argument |
|---|---|
| `partitionBy("customer_id")` | `keys=["customer_id"]` |
| `orderBy(seq desc)`, `rn = 1` | `sequence_by=F.col("seq")` |
| `is_deleted` tombstone row, kept | `apply_as_deletes=F.expr("op = 'D'")` |
| `tomb_batches` window | `pipelines.cdc.tombstoneGCThresholdInSeconds` |
| dropping `op`, `seq`, `arrival` from the output | `except_column_list=[...]` |
| unioning state + batch each time | the streaming table itself is the state |

The production version of Cases C is `silver_customers_cdc_pipeline.py`.

In [11]:
assert Cc.filter("not is_deleted").count() == truth["expected_scd1_final_rows"] == 8170
assert Cc.filter("is_deleted").count() == truth["expected_deleted_and_not_recreated"] == 130
assert score(Cc)[1:] == (0, 0)
print("All good: the from-scratch replay matches the answer key.")

All good: the from-scratch replay matches the answer key.


## 6. Limits

- Arrival is simulated by file; a real stream also reorders within a file.
- State is recomputed from a union each batch. Fine for 8,000 rows, not for production, where Delta `MERGE` or the pipeline engine does it incrementally.
- No SCD2 here. A version history needs `__START_AT`/`__END_AT`, which the pipeline file builds.